# Bygging med Meta-familie-modeller  

## Introduksjon  

Denne leksjonen vil dekke:  

- Utforsking av de to hovedmodellene i Meta-familien - Llama 3.1 og Llama 3.2  
- Forstå brukstilfeller og scenarioer for hver modell  
- Kodeeksempel som viser de unike funksjonene til hver modell  


## Meta-familien av modeller  

I denne leksjonen skal vi utforske 2 modeller fra Meta-familien eller "Llama-flokken" - Llama 3.1 og Llama 3.2  

Disse modellene finnes i forskjellige varianter og er tilgjengelige i [Microsoft Foundry Models-katalogen](https://ai.azure.com/catalog/models?WT.mc_id=academic-105485-koreyst).  

> **Merk:** GitHub Models fases ut ved slutten av juli 2026. Her er flere detaljer om bruk av [Microsoft Foundry Models](https://learn.microsoft.com/en-us/azure/ai-foundry/model-inference/overview?WT.mc_id=academic-105485-koreyst) for å prototype med AI-modeller.  

Modellvarianter:  
- Llama 3.1 - 70B Instruct  
- Llama 3.1 - 405B Instruct  
- Llama 3.2 - 11B Vision Instruct  
- Llama 3.2 - 90B Vision Instruct  

*Merk: Llama 3 er også tilgjengelig i Microsoft Foundry Models men vil ikke bli dekket i denne leksjonen*  



## Llama 3.1 

Med 405 milliarder parametere, passer Llama 3.1 inn i kategorien åpen kildekode LLM. 

Modellen er en oppgradering av den tidligere utgivelsen Llama 3 ved å tilby: 

- Større kontekstvindu - 128k tokens vs 8k tokens 
- Større maks antall output-tokens - 4096 vs 2048 
- Bedre flerspråklig støtte - på grunn av økt antall trenings-tokens 

Dette gjør at Llama 3.1 kan håndtere mer komplekse bruksområder når man bygger GenAI-applikasjoner, inkludert: 
- Innfødt funksjonsanrop - muligheten til å kalle eksterne verktøy og funksjoner utenfor LLM-arbeidsflyten
- Bedre RAG-ytelse - på grunn av det større kontekstvinduet 
- Syntetisk datagenerering - muligheten til å lage effektiv data for oppgaver som finjustering 



### Naturlig funksjonskalling 

Llama 3.1 har blitt finjustert for å være mer effektiv i å utføre funksjons- eller verktøykall. Den har også to innebygde verktøy som modellen kan identifisere som nødvendige å bruke basert på brukerens prompt. Disse verktøyene er: 

- **Brave Search** - Kan brukes for å hente oppdatert informasjon som været ved å utføre et nettsøk 
- **Wolfram Alpha** - Kan brukes for mer komplekse matematiske beregninger, så det er ikke nødvendig å skrive egne funksjoner. 

Du kan også lage dine egne tilpassede verktøy som LLM kan kalle. 

I kodeeksempelet nedenfor: 

- Definerer vi tilgjengelige verktøy (brave_search, wolfram_alpha) i system-prompten. 
- Sender en brukerprompt som spør om været i en bestemt by. 
- LLM vil svare med et verktøykall til Brave Search-verktøyet som vil se slik ut `<|python_tag|>brave_search.call(query="Stockholm weather")` 

*Merk: Dette eksempelet utfører kun verktøykallet, hvis du ønsker å få resultatene, må du opprette en gratis konto på Brave API-siden og definere funksjonen selv* 


In [ ]:
%pip install azure-core
%pip install azure-ai-inference

In [ ]:
import os
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import AssistantMessage, SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential

# Get these from your Microsoft Foundry project's "Overview" page
token = os.environ["AZURE_INFERENCE_CREDENTIAL"]
endpoint = os.environ["AZURE_INFERENCE_ENDPOINT"]
model_name = "Meta-Llama-3.1-405B-Instruct"

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)


tool_prompt=f"""
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Environment: ipython
Tools: brave_search, wolfram_alpha
Cutting Knowledge Date: December 2023
Today Date: 23 July 2024

You are a helpful assistant<|eot_id|>
"""

messages = [
    SystemMessage(content=tool_prompt),
    UserMessage(content="What is the weather in Stockholm?"),

]

response = client.complete(messages=messages, model=model_name)

print(response.choices[0].message.content)






### Llama 3.2 

Selv om det er et LLM, har Llama 3.1 en begrensning, nemlig multimodalitet. Det vil si å kunne bruke forskjellige typer input som bilder som prompt og gi svar. Denne evnen er en av hovedfunksjonene i Llama 3.2. Disse funksjonene inkluderer også: 

- Multimodalitet - har evnen til å evaluere både tekst- og bildeprompt 
- Varianter i små til mellomstore størrelser (11B og 90B) - dette gir fleksible distribusjonsmuligheter, 
- Tekstbare varianter (1B og 3B) - dette gjør det mulig å distribuere modellen på edge / mobil enheter og gir lav ventetid 

Støtten for multimodalitet representerer et stort steg i verden av åpen kildekode-modeller. Kodeeksemplet nedenfor tar både et bilde og tekstprompt for å få en analyse av bildet fra Llama 3.2 90B. 

### Multimodal støtte med Llama 3.2


In [ ]:
%pip install azure-core
%pip install azure-ai-inference

In [ ]:
import os
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import (
    SystemMessage,
    UserMessage,
    TextContentItem,
    ImageContentItem,
    ImageUrl,
    ImageDetailLevel,
)
from azure.core.credentials import AzureKeyCredential

# Get these from your Microsoft Foundry project's "Overview" page
token = os.environ["AZURE_INFERENCE_CREDENTIAL"]
endpoint = os.environ["AZURE_INFERENCE_ENDPOINT"]
model_name = "Llama-3.2-90B-Vision-Instruct"


client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)

response = client.complete(
    messages=[
        SystemMessage(
            content="You are a helpful assistant that describes images in details."
        ),
        UserMessage(
            content=[
                TextContentItem(text="What's in this image?"),
                ImageContentItem(
                    image_url=ImageUrl.load(
                        image_file="sample.jpg",
                        image_format="jpg",
                        detail=ImageDetailLevel.LOW)
                ),
            ],
        ),
    ],
    model=model_name,
)

print(response.choices[0].message.content)


## Læringen stopper ikke her, fortsett reisen

Etter å ha fullført denne leksjonen, sjekk ut vår [Generative AI Learning collection](https://aka.ms/genai-collection?WT.mc_id=academic-105485-koreyst) for å fortsette å øke kunnskapen din om Generativ AI!


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Ansvarsfraskrivelse**:
Dette dokumentet er oversatt ved hjelp av AI-oversettelsestjenesten [Co-op Translator](https://github.com/Azure/co-op-translator). Selv om vi streber etter nøyaktighet, vær oppmerksom på at automatiske oversettelser kan inneholde feil eller unøyaktigheter. Det opprinnelige dokumentet på originalspråket skal betraktes som den autoritative kilden. For kritisk informasjon anbefales profesjonell menneskelig oversettelse. Vi er ikke ansvarlige for eventuelle misforståelser eller feiltolkninger som oppstår ved bruk av denne oversettelsen.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
